# PS04 - Comparing Tree Methods
**Decision Trees vs. Random Forests vs. Gradient Boosting**

This assignment works through the entire modeling process, comparing the different tree models in terms of performance, bias and variance.  

We'll be using US Census data to predict, based on other attributes, whether an individual has a yearly income above or below $50 K (a binary classification).

**Process Roadmap**  

0. Load and explore the data
1. Pre-process (ordinal + one-hot encoding, imputation, train/test split)
2. Modeling x3
3. Model Assessment, Head-to-head performance comparison
4. Interpretation

In [30]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

## 0. Load & Explore

In [23]:
## fetching the data
adult, income = fetch_openml('adult', version=2, as_frame=True, return_X_y = True, parser='auto')

In [24]:
adult.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States


In [25]:
income.head()

0    <=50K
1    <=50K
2     >50K
3     >50K
4    <=50K
Name: class, dtype: category
Categories (2, object): ['<=50K', '>50K']

In [26]:
income.value_counts()

class
<=50K    37155
>50K     11687
Name: count, dtype: int64

## Inspect the data

- Do we have missing values? How should we deal with them?
- What is the split of over- and under-50K in our data?
- Are there features we have questions about?


## 2. Pre-processing

**Imputation**
In the past, when we've had missing data, we've discarded the entire row (sample). In this example, we'll introduce *imputation*, inferring a value for missing data. And we'll start with the simplest form of imputation, replacing data with a constant value.

The transformer we use is called [`SimpleImputer`](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html) and it permits several options for filling in missing data, selectable through the `strategy` hyperparameter:

 - `mean` - the mean of available data
 - `median` - the median of available data
 - `most_frequent` - the mode of available data
 - `constant` - a user-selected value (must then set `fill_value`)

We'll use 'median'.

**Encoding**
Some columns will require encoding and we'll apply `OrdinalEncoder` or `OneHotEncoder` when appropriate.

In [27]:
X_train, X_test, y_train, y_test = train_test_split(adult_df, income, test_size=0.20, random_state=42)

## 3. Modeling and Model Comparison

Let's try two comparisons, sweeping max_depth while keeping forest size at 100 and sweeping forest size while keeping max depth at 2.

- **Decision Tree** → rapidly overfits: high variance at deep depths, high bias at shallow depths.  
- **Random Forest** → averaging over many trees *reduces variance* without increasing bias much.  
- **Gradient Boosting** → each stage corrects the residual of the previous one, *reducing bias* step-by-step.

For each model, calculate and store the accuracy on the training and testing sets.

In [28]:
### Sweep max_depth
depths = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10,  15, 20])

In [29]:
### Sweep n_estimators
n_est = np.array([2, 4, 8, 16, 32, 64, 128, 256, 512, 1028])

## 4. Visualization

Create two sets of subplots, one for each hyper-parameter sweep. Each subplot corresponds to one of the model types.

For each of these sweeps and for each model, plot a line graph showing how accuracy changes as the hyperparameter is changed, comparing trends between the training and testing set.

## Summary

| Property | Decision Tree | Random Forest | Gradient Boosting |
|---|:---:|:---:|:---:|
| **Bias** | Medium (tunable) | Medium | **Low** |
| **Variance** | **High** (overfit easily) | **Low** (averaging) | Low–Medium |
| **Interpretability** | ✅ High | ⚠️ Medium | ⚠️ Low |
| **Training speed** | ✅ Very fast | Medium | ⚠️ Slower |
| **Prediction speed** | ✅ Very fast | Medium | Medium |
| **Peak accuracy** | Lowest | Middle | **Highest** |

### Key Takeaways

1. **Decision Trees** are powerful building blocks but suffer from **high variance**. A tiny change to training data can completely alter the tree structure. 

2. **Random Forests** lessens variance by **averaging many independently-grown trees** (each trained on a random bootstrap + random feature subset). Even with deep, unconstrained trees, the ensemble generalises well. A commonly accepted belief in machine learning circles states that *you can't overfit a random forest*.

3. **Gradient Boosting** aims at reducing **bias**: each new shallow tree is fit to the *residuals* of the previous ensemble. The model starts with high bias and systematically reduces it. The cost is paid as more training time and a larger number of hyperparameters.

4. **Feature encoding** matters: the categorical columns (`education`, `occupation`, `workclass`, `relationship`, `marital-status`, `race`, `sex`, `native-country`) were one-hot encoded and contributed substantially to model performance, as confirmed by RF feature importances.

> **Practical advice:** start with a Random Forest (robust, few hyperparameters), then try Gradient Boosting (XGBoost/LightGBM in production) if you need maximum performance. Use a Decision Tree when interpretability is a hard requirement.